# UK Logistics & Road Freight Intelligence Dashboard

## Notebook 03: Exploratory Data Analysis

This notebook explores the cleaned Department for Transport road freight and road traffic datasets.

The purpose of this notebook is to:

- Review all processed datasets
- Identify useful fields for logistics analysis
- Analyse row counts, column counts and numeric measures
- Find datasets containing year, region, vehicle, road type and freight measures
- Create summary tables for SQL and Power BI
- Export dashboard-ready exploratory outputs

This stage helps decide which datasets are strongest for the final UK Logistics & Road Freight Intelligence Dashboard.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [2]:
# Define paths

processed_data_path = Path("../data/processed")

print("Processed data folder exists:", processed_data_path.exists())

Processed data folder exists: True


In [3]:
# List processed CSV files

processed_files = sorted([
    file for file in processed_data_path.iterdir()
    if file.is_file() and file.suffix.lower() == ".csv"
])

print("Processed CSV files found:", len(processed_files))

for file in processed_files:
    print(file.name)

Processed CSV files found: 5
cleaned_local_authority_traffic.csv
cleaned_region_traffic_by_road_type.csv
cleaned_region_traffic_by_vehicle_type.csv
processed_file_inventory.csv
raw_file_inventory.csv


In [4]:
# Load processed file inventory if available

inventory_file = processed_data_path / "processed_file_inventory.csv"

if inventory_file.exists():
    processed_inventory = pd.read_csv(inventory_file)
    display(processed_inventory)
else:
    print("Processed inventory file not found.")

,file_name,file_size_mb,estimated_rows,columns,column_names
0,cleaned_local_authority_traffic.csv,0.44,6560,8,"local_authority_id, local_authority_name, loca..."
1,cleaned_region_traffic_by_road_type.csv,0.10,1623,9,"year, region_id, region_name, region_ons_code,..."
2,cleaned_region_traffic_by_vehicle_type.csv,0.05,352,13,"year, region_id, region_name, region_ons_code,..."


In [5]:
# Create profile summary for all processed CSV files

dataset_profiles = []

for file in processed_files:
    try:
        df = pd.read_csv(file, low_memory=False)
        
        dataset_profiles.append({
            "file_name": file.name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "file_size_mb": round(file.stat().st_size / (1024 * 1024), 2),
            "column_names": ", ".join(df.columns.tolist()[:15])
        })
        
    except Exception as error:
        dataset_profiles.append({
            "file_name": file.name,
            "rows": "error",
            "columns": "error",
            "file_size_mb": round(file.stat().st_size / (1024 * 1024), 2),
            "column_names": str(error)
        })

dataset_profiles_df = pd.DataFrame(dataset_profiles)

dataset_profiles_df

,file_name,rows,columns,file_size_mb,column_names
0,cleaned_local_authority_traffic.csv,6560,8,0.44,"local_authority_id, local_authority_name, loca..."
1,cleaned_region_traffic_by_road_type.csv,1623,9,0.10,"year, region_id, region_name, region_ons_code,..."
2,cleaned_region_traffic_by_vehicle_type.csv,352,13,0.05,"year, region_id, region_name, region_ons_code,..."
3,processed_file_inventory.csv,3,5,0.00,"file_name, file_size_mb, estimated_rows, colum..."
4,raw_file_inventory.csv,8,3,0.00,"file_name, extension, size_mb"


## Search for Useful Logistics Columns

This section searches across all processed datasets for columns related to year, region, vehicle type, road type, traffic, freight, goods moved, goods lifted and distance travelled.

In [6]:
# Search for useful columns across processed files

keywords = [
    "year",
    "region",
    "country",
    "local",
    "authority",
    "vehicle",
    "road",
    "traffic",
    "hgv",
    "goods",
    "tonne",
    "kilometre",
    "kilometer",
    "distance",
    "lifted",
    "moved",
    "empty",
    "loading",
    "factor"
]

column_search_results = []

for file in processed_files:
    try:
        df = pd.read_csv(file, nrows=5, low_memory=False)
        
        for column in df.columns:
            column_lower = column.lower()
            
            matched_keywords = [
                keyword for keyword in keywords
                if keyword in column_lower
            ]
            
            if matched_keywords:
                column_search_results.append({
                    "file_name": file.name,
                    "column_name": column,
                    "matched_keywords": ", ".join(matched_keywords)
                })
                
    except Exception as error:
        column_search_results.append({
            "file_name": file.name,
            "column_name": "ERROR",
            "matched_keywords": str(error)
        })

column_search_df = pd.DataFrame(column_search_results)

column_search_df

,file_name,column_name,matched_keywords
0,cleaned_local_authority_traffic.csv,local_authority_id,"local, authority"
1,cleaned_local_authority_traffic.csv,local_authority_name,"local, authority"
2,cleaned_local_authority_traffic.csv,local_authority_code,"local, authority"
3,cleaned_local_authority_traffic.csv,year,year
4,cleaned_local_authority_traffic.csv,all_motor_vehicles,vehicle
5,cleaned_region_traffic_by_road_type.csv,year,year
6,cleaned_region_traffic_by_road_type.csv,region_id,region
7,cleaned_region_traffic_by_road_type.csv,region_name,region
8,cleaned_region_traffic_by_road_type.csv,region_ons_code,region
9,cleaned_region_traffic_by_road_type.csv,road_category_id,road


In [7]:
# Score files based on useful logistics keywords

file_scores = []

for file in processed_files:
    try:
        df = pd.read_csv(file, nrows=5, low_memory=False)
        columns_text = " ".join(df.columns.astype(str).str.lower().tolist())
        
        score = sum(keyword in columns_text for keyword in keywords)
        
        file_scores.append({
            "file_name": file.name,
            "rows_estimate": "see inventory",
            "columns": df.shape[1],
            "keyword_score": score,
            "column_names": ", ".join(df.columns.tolist()[:12])
        })
        
    except Exception as error:
        file_scores.append({
            "file_name": file.name,
            "rows_estimate": "error",
            "columns": "error",
            "keyword_score": 0,
            "column_names": str(error)
        })

file_scores_df = pd.DataFrame(file_scores).sort_values(
    by="keyword_score",
    ascending=False
)

file_scores_df

,file_name,rows_estimate,columns,keyword_score,column_names
0,cleaned_local_authority_traffic.csv,see inventory,8,4,"local_authority_id, local_authority_name, loca..."
1,cleaned_region_traffic_by_road_type.csv,see inventory,9,4,"year, region_id, region_name, region_ons_code,..."
2,cleaned_region_traffic_by_vehicle_type.csv,see inventory,13,4,"year, region_id, region_name, region_ons_code,..."
3,processed_file_inventory.csv,see inventory,5,0,"file_name, file_size_mb, estimated_rows, colum..."
4,raw_file_inventory.csv,see inventory,3,0,"file_name, extension, size_mb"


In [8]:
# Create numeric summaries for processed datasets

numeric_summary_rows = []

for file in processed_files:
    try:
        df = pd.read_csv(file, low_memory=False)
        
        numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
        
        for column in numeric_columns:
            numeric_summary_rows.append({
                "file_name": file.name,
                "numeric_column": column,
                "non_null_count": df[column].notna().sum(),
                "min_value": df[column].min(),
                "max_value": df[column].max(),
                "mean_value": df[column].mean()
            })
            
    except Exception as error:
        numeric_summary_rows.append({
            "file_name": file.name,
            "numeric_column": "ERROR",
            "non_null_count": None,
            "min_value": None,
            "max_value": None,
            "mean_value": str(error)
        })

numeric_summary_df = pd.DataFrame(numeric_summary_rows)

numeric_summary_df.head(50)

,file_name,numeric_column,non_null_count,min_value,max_value,mean_value
0,cleaned_local_authority_traffic.csv,local_authority_id,6560,1.000000e+00,2.140000e+02,1.041192e+02
1,cleaned_local_authority_traffic.csv,year,6560,1.993000e+03,2.024000e+03,2.008540e+03
2,cleaned_local_authority_traffic.csv,link_length_km,6560,3.418000e+01,1.313417e+04,1.916357e+03
3,cleaned_local_authority_traffic.csv,link_length_miles,6560,2.124000e+01,8.161190e+03,1.190769e+03
4,cleaned_local_authority_traffic.csv,cars_and_taxis,6560,6.000000e+05,7.811400e+09,1.165554e+09
5,cleaned_local_authority_traffic.csv,all_motor_vehicles,6560,1.200000e+06,9.849700e+09,1.477510e+09
6,cleaned_region_traffic_by_road_type.csv,year,1623,1.993000e+03,2.024000e+03,2.008935e+03
7,cleaned_region_traffic_by_road_type.csv,region_id,1623,1.000000e+00,1.100000e+01,6.162046e+00
8,cleaned_region_traffic_by_road_type.csv,road_category_id,1623,1.000000e+00,5.000000e+00,3.076402e+00
9,cleaned_region_traffic_by_road_type.csv,link_length_km,1623,0.000000e+00,4.896410e+04,7.745717e+03


In [9]:
# Create a dashboard candidate summary

dashboard_candidate_summary = file_scores_df.merge(
    dataset_profiles_df[["file_name", "rows", "file_size_mb"]],
    on="file_name",
    how="left"
)

dashboard_candidate_summary = dashboard_candidate_summary[
    [
        "file_name",
        "rows",
        "columns",
        "file_size_mb",
        "keyword_score",
        "column_names"
    ]
].sort_values(by=["keyword_score", "rows"], ascending=False)

dashboard_candidate_summary

,file_name,rows,columns,file_size_mb,keyword_score,column_names
0,cleaned_local_authority_traffic.csv,6560,8,0.44,4,"local_authority_id, local_authority_name, loca..."
1,cleaned_region_traffic_by_road_type.csv,1623,9,0.10,4,"year, region_id, region_name, region_ons_code,..."
2,cleaned_region_traffic_by_vehicle_type.csv,352,13,0.05,4,"year, region_id, region_name, region_ons_code,..."
4,raw_file_inventory.csv,8,3,0.00,0,"file_name, extension, size_mb"
3,processed_file_inventory.csv,3,5,0.00,0,"file_name, file_size_mb, estimated_rows, colum..."


In [10]:
# Export EDA metadata outputs

dataset_profiles_df.to_csv(
    processed_data_path / "eda_dataset_profiles.csv",
    index=False
)

column_search_df.to_csv(
    processed_data_path / "eda_column_search_results.csv",
    index=False
)

numeric_summary_df.to_csv(
    processed_data_path / "eda_numeric_summary.csv",
    index=False
)

dashboard_candidate_summary.to_csv(
    processed_data_path / "eda_dashboard_candidate_summary.csv",
    index=False
)

print("EDA metadata outputs exported successfully.")

EDA metadata outputs exported successfully.


In [11]:
# Preview top candidate files

top_candidate_files = dashboard_candidate_summary["file_name"].head(10).tolist()

for file_name in top_candidate_files:
    file_path = processed_data_path / file_name
    
    print("\n" + "=" * 100)
    print("Candidate file:", file_name)
    print("=" * 100)
    
    try:
        df_preview = pd.read_csv(file_path, nrows=10, low_memory=False)
        print("Shape preview:", df_preview.shape)
        print("Columns:", df_preview.columns.tolist())
        display(df_preview.head(10))
        
    except Exception as error:
        print("Could not preview:", file_name)
        print("Error:", error)


Candidate file: cleaned_local_authority_traffic.csv
Shape preview: (10, 8)
Columns: ['local_authority_id', 'local_authority_name', 'local_authority_code', 'year', 'link_length_km', 'link_length_miles', 'cars_and_taxis', 'all_motor_vehicles']


,local_authority_id,local_authority_name,local_authority_code,year,link_length_km,link_length_miles,cars_and_taxis,all_motor_vehicles
0,1,Isles of Scilly,E06000053,1993,36.01,22.38,9.000000e+05,1.400000e+06
1,2,Nottinghamshire,E10000024,1993,4650.69,2889.80,2.702600e+09,3.368900e+09
2,3,Glasgow City,S12000049,1993,1768.54,1098.92,1.381200e+09,1.662600e+09
3,4,North Lanarkshire,S12000050,1993,1606.71,998.36,1.253500e+09,1.547600e+09
4,5,Somerset,E06000066,1993,6630.33,4119.90,2.410300e+09,2.958200e+09
5,6,Newport,W06000022,1993,708.92,440.50,7.123000e+08,8.832000e+08
6,7,Bridgend,W06000013,1993,731.80,454.72,5.042000e+08,5.991000e+08
7,8,Swansea,W06000011,1993,1139.55,708.08,7.119000e+08,8.387000e+08
8,9,Isle of Anglesey,W06000001,1993,1154.03,717.08,2.174000e+08,2.647000e+08
9,10,Gwynedd,W06000002,1993,2405.76,1494.87,5.245000e+08,6.397000e+08



Candidate file: cleaned_region_traffic_by_road_type.csv
Shape preview: (10, 9)
Columns: ['year', 'region_id', 'region_name', 'region_ons_code', 'road_category_id', 'road_category_name', 'link_length_km', 'link_length_miles', 'all_motor_vehicles']


,year,region_id,region_name,region_ons_code,road_category_id,road_category_name,link_length_km,link_length_miles,all_motor_vehicles
0,1993,1,South West,E12000009,1,TM,301.34,187.24,3.465800e+09
1,1993,1,South West,E12000009,3,TA,993.59,617.39,3.484700e+09
2,1993,1,South West,E12000009,4,PA,3874.92,2407.76,7.794000e+09
3,1993,1,South West,E12000009,5,M,43581.70,27080.41,9.112000e+09
4,1993,2,East Midlands,E12000004,1,TM,178.61,110.98,2.736700e+09
5,1993,2,East Midlands,E12000004,3,TA,1219.23,757.59,4.936700e+09
6,1993,2,East Midlands,E12000004,4,PA,2571.21,1597.68,5.535700e+09
7,1993,2,East Midlands,E12000004,5,M,26712.70,16598.50,7.083400e+09
8,1993,3,Scotland,S92000003,1,TM,335.26,208.32,2.485500e+09
9,1993,3,Scotland,S92000003,3,TA,2820.30,1752.45,5.052900e+09



Candidate file: cleaned_region_traffic_by_vehicle_type.csv
Shape preview: (10, 13)
Columns: ['year', 'region_id', 'region_name', 'region_ons_code', 'link_length_km', 'link_length_miles', 'pedal_cycles', 'two_wheeled_motor_vehicles', 'cars_and_taxis', 'buses_and_coaches', 'lgvs', 'all_hgvs', 'all_motor_vehicles']


,year,region_id,region_name,region_ons_code,link_length_km,link_length_miles,pedal_cycles,two_wheeled_motor_vehicles,cars_and_taxis,buses_and_coaches,lgvs,all_hgvs,all_motor_vehicles
0,1993,1,South West,E12000009,48751.55,30292.81,228000000.0,289900000.0,1.967430e+10,245000000.0,2.351700e+09,1.295600e+09,2.385660e+10
1,1993,2,East Midlands,E12000004,30681.75,19064.76,199700000.0,172800000.0,1.620020e+10,190100000.0,2.082200e+09,1.647100e+09,2.029240e+10
2,1993,3,Scotland,S92000003,58532.65,36370.50,147300000.0,126400000.0,1.767760e+10,334300000.0,2.231000e+09,1.339900e+09,2.170930e+10
3,1993,4,Wales,W92000004,32543.50,20221.59,81700000.0,104600000.0,1.127880e+10,156300000.0,1.452700e+09,7.352000e+08,1.372760e+10
4,1993,5,North West,E12000002,35952.60,22339.91,251400000.0,199200000.0,2.373000e+10,339800000.0,2.801800e+09,1.834400e+09,2.890520e+10
5,1993,6,London,E12000007,14350.45,8916.96,272100000.0,358200000.0,1.588250e+10,273400000.0,1.928600e+09,6.409000e+08,1.908350e+10
6,1993,7,East of England,E12000006,38708.43,24052.30,365600000.0,270100000.0,2.299610e+10,254800000.0,2.976400e+09,1.741100e+09,2.823840e+10
7,1993,8,Yorkshire and the Humber,E12000003,31096.60,19322.53,231400000.0,171200000.0,1.687600e+10,235900000.0,2.204300e+09,1.624300e+09,2.111170e+10
8,1993,9,South East,E12000008,46587.54,28948.16,431700000.0,415200000.0,3.674830e+10,368700000.0,4.267100e+09,2.035300e+09,4.383460e+10
9,1993,10,West Midlands,E12000005,31813.53,19768.01,195700000.0,177400000.0,2.048420e+10,298600000.0,2.528700e+09,1.703300e+09,2.519230e+10



Candidate file: raw_file_inventory.csv
Shape preview: (8, 3)
Columns: ['file_name', 'extension', 'size_mb']


,file_name,extension,size_mb
0,.gitkeep,NaN,0.00
1,local_authority_traffic.csv,.csv,0.43
2,region_traffic_by_road_type.csv,.csv,0.10
3,region_traffic_by_vehicle_type.csv,.csv,0.04
4,rfs0101.ods,.ods,0.01
5,rfs0121.ods,.ods,0.01
6,rfs0122.ods,.ods,0.01
7,rfs0125.ods,.ods,0.01



Candidate file: processed_file_inventory.csv
Shape preview: (3, 5)
Columns: ['file_name', 'file_size_mb', 'estimated_rows', 'columns', 'column_names']


,file_name,file_size_mb,estimated_rows,columns,column_names
0,cleaned_local_authority_traffic.csv,0.44,6560,8,"local_authority_id, local_authority_name, loca..."
1,cleaned_region_traffic_by_road_type.csv,0.10,1623,9,"year, region_id, region_name, region_ons_code,..."
2,cleaned_region_traffic_by_vehicle_type.csv,0.05,352,13,"year, region_id, region_name, region_ons_code,..."


## Exploratory Data Analysis Notes

This notebook reviewed the cleaned Department for Transport datasets and created metadata outputs to support the next stage.

Completed actions:

- Reviewed all processed CSV files
- Created dataset profile summary
- Searched for useful logistics columns
- Identified likely dashboard candidate datasets
- Created numeric summaries
- Exported EDA metadata outputs
- Previewed the top candidate files

The next stage will create selected analysis-ready datasets for SQL and Power BI. These will focus on road freight activity, HGV/vehicle traffic, regional traffic and logistics demand patterns.